# Bot Workflow Runs Summary Extractor

**Date:** 2026-02-09  
**Purpose:** Extract key information from GitHub Actions workflow runs and create Excel summary  
**Version:** 007f - COMPLETE with question type classification

This notebook:
1. Fetches workflow runs from GitHub Actions API (handles pagination correctly)
2. Downloads logs for each run (robust Unicode handling)
3. Extracts: run number, timestamp, question presence, question number, **question type**, forecast value, error flag
4. Outputs to **VERSIONED** Excel files: `Runs and Question Numbers_v001.xlsx`, `_v002.xlsx`, etc.

**ALL FIXES:**
- ✅ ANSI color code stripping
- ✅ Pagination (13 pages, 1224 runs)
- ✅ UTF-8 byte-based decoding
- ✅ Timestamp extraction (space format not ISO)
- ✅ **FORECAST EXTRACTION (pattern-based, not type-based)**
- ✅ **QUESTION TYPE DETECTION** (Binary, Numeric, Multiple Choice)
- ✅ **HANDLES TIMESTAMP PREFIXES in logs**
- ✅ **EXCEL VERSIONING** (no duplicates!)

---

In [ ]:
# Imports
import json
import re
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, Alignment

print("✅ Imports successful")

In [ ]:
# Configuration
REPO = "D-Enns/metac-bot-template"
WORKFLOW = "dre_run_bot_on_tournament.yaml"
OUTPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Runs and Question Numbers.xlsx")
TEST_LIMIT = 10  # Set to None for all runs

print(f"Repository: {REPO}")
print(f"Workflow: {WORKFLOW}")
print(f"Output base: {OUTPUT_FILE}")
print(f"Test limit: {TEST_LIMIT}")

In [ ]:
# Data structure
@dataclass
class RunInfo:
    workflow_run_number: int
    time_date: str
    has_question: str
    question_number: str
    question_type: str
    forecast_value: str
    error_flag: str

print("✅ RunInfo defined")

In [ ]:
# Utility: Strip ANSI codes
def strip_ansi_codes(text: str) -> str:
    """Remove ANSI color codes."""
    return re.sub(r'\x1b\[[0-9;]*m', '', text)

print("✅ strip_ansi_codes() defined")

In [ ]:
# GitHub CLI: Verify authentication
def verify_gh_cli() -> bool:
    try:
        result = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True, timeout=10)
        if result.returncode != 0:
            print("❌ gh CLI not authenticated")
            return False
        print("✅ GitHub CLI authenticated")
        return True
    except FileNotFoundError:
        print("❌ gh CLI not installed")
        return False

verify_gh_cli()

In [ ]:
# GitHub CLI: Fetch workflow runs with pagination
def get_workflow_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[Dict]:
    print(f"Fetching workflow runs...")
    cmd = ["gh", "api", f"repos/{repo}/actions/workflows/{workflow}/runs", "--paginate"]
    
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if result.returncode != 0:
        print(f"❌ Failed: {result.stderr}")
        return []
    
    clean = strip_ansi_codes(result.stdout)
    
    # Parse multiple JSON objects by counting braces
    json_objects = []
    current = ""
    count = 0
    
    for char in clean:
        current += char
        if char == '{':
            count += 1
        elif char == '}':
            count -= 1
            if count == 0 and current.strip():
                json_objects.append(current.strip())
                current = ""
    
    print(f"Found {len(json_objects)} pages")
    
    runs = []
    for obj_str in json_objects:
        data = json.loads(obj_str)
        if 'workflow_runs' in data:
            runs.extend(data['workflow_runs'])
    
    simplified = [{'id': r['id'], 'run_number': r['run_number'], 
                   'created_at': r['created_at'], 'conclusion': r.get('conclusion', '')} 
                  for r in runs]
    simplified.sort(key=lambda x: x['run_number'], reverse=True)
    
    if limit:
        simplified = simplified[:limit]
    
    print(f"✅ {len(simplified)} runs")
    return simplified

test_runs = get_workflow_runs(REPO, WORKFLOW, limit=5)
print(f"Sample: Run #{test_runs[0]['run_number']}" if test_runs else "No runs")

In [ ]:
# GitHub CLI: Download log (byte-based for Unicode)
def download_run_log(repo: str, run_id: int, run_number: int) -> Optional[str]:
    cmd = ["gh", "run", "view", str(run_id), "--repo", repo, "--log"]
    try:
        result = subprocess.run(cmd, capture_output=True, timeout=120)
        if result.returncode != 0:
            return None
        
        # Decode bytes manually
        try:
            log_text = result.stdout.decode('utf-8', errors='replace')
        except:
            log_text = result.stdout.decode('latin-1', errors='replace')
        
        return strip_ansi_codes(log_text)
    except Exception as e:
        print(f"  Error: {e}")
        return None

print("✅ download_run_log() defined")

In [ ]:
# Regex patterns
QUESTION_URL_PATTERN = re.compile(r'https://www\.metaculus\.com/questions/(\d+)')
FOUND_RESEARCH_PATTERN = re.compile(r'Found Research for URL')
ERROR_EXIT_PATTERN = re.compile(r'Error: Process completed with exit code 1\.')

print("✅ Patterns defined")

In [ ]:
# Extract timestamp (Python logger format: space not T)
def extract_timestamp(log: str) -> str:
    match = re.search(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', log)
    return match.group(1) if match else ""

print("✅ extract_timestamp()")

In [ ]:
# Check if has question
def check_has_question(log: str) -> str:
    return "Y" if FOUND_RESEARCH_PATTERN.search(log) else "N"

def extract_question_number(log: str) -> str:
    match = QUESTION_URL_PATTERN.search(log)
    return match.group(1) if match else ""

def check_error_flag(log: str) -> str:
    last_lines = '\n'.join(log.split('\n')[-200:])
    return "Y" if ERROR_EXIT_PATTERN.search(last_lines) else "N"

print("✅ Check functions defined")

In [ ]:
# Extract forecasts - pattern-based (handles timestamp prefixes)
def extract_binary_forecast(log: str) -> str:
    patterns = [
        r'## Final Prediction[^\n]*\n[^\n]*?(\d+\.?\d*)%?',  # Two hashes
        r'### Final Prediction[^\n]*\n[^\n]*?(\d+\.?\d*)%?', # Three hashes
        r'\*Final Prediction\*:\s*(\d+\.?\d*)%?',
        r'\*\*Probability:\s*(\d+\.?\d*)%?'
    ]
    for p in patterns:
        m = re.search(p, log)
        if m:
            return m.group(1)
    raise ValueError("Binary not found")

def extract_mc_forecast(log: str) -> str:
    # Pattern handles timestamp prefixes before the dict
    for p in [r'## Final Answer[^\n]*\n[^\{]*(\{[^}]+\})',  # Two hashes
              r'### Final Answer[^\n]*\n[^\{]*(\{[^}]+\})', # Three hashes
              r'# Final Answer[^\n]*\n[^\{]*(\{[^}]+\})']:  # One hash
        m = re.search(p, log)
        if m:
            return m.group(1)
    raise ValueError("MC not found")

def extract_numeric_forecast(log: str) -> str:
    # Pattern handles timestamp prefixes before the array: [^\[]* matches anything before [
    for p in [r'## Final Answer[^\n]*\n[^\[]*(\[[\d.,\s]+\])',  # Two hashes
              r'### Final Answer[^\n]*\n[^\[]*(\[[\d.,\s]+\])', # Three hashes
              r'# Final Answer[^\n]*\n[^\[]*(\[[\d.,\s]+\])']:  # One hash
        m = re.search(p, log)
        if m:
            return m.group(1)
    raise ValueError("Numeric not found")

def extract_forecast_value(log: str) -> Tuple[str, str]:
    """Try all extraction methods and return (forecast_value, question_type)."""
    # Try numeric first (most common with Points)
    try:
        forecast = extract_numeric_forecast(log)
        return (forecast, "Numeric")
    except:
        pass
    
    # Try binary
    try:
        forecast = extract_binary_forecast(log)
        return (forecast, "Binary")
    except:
        pass
    
    # Try multiple choice
    try:
        forecast = extract_mc_forecast(log)
        return (forecast, "Multiple Choice")
    except:
        pass
    
    return ("", "")  # No forecast found

print("✅ Forecast extraction functions")

In [ ]:
# Process single log
def process_log(log: str, run_number: int) -> RunInfo:
    time_date = extract_timestamp(log)
    has_question = check_has_question(log)
    question_number = extract_question_number(log) if has_question == "Y" else ""
    
    forecast_value = ""
    question_type = ""
    if has_question == "Y":
        forecast_value, question_type = extract_forecast_value(log)
    
    error_flag = check_error_flag(log)
    
    return RunInfo(
        workflow_run_number=run_number,
        time_date=time_date,
        has_question=has_question,
        question_number=question_number,
        question_type=question_type,
        forecast_value=forecast_value,
        error_flag=error_flag
    )

print("✅ process_log()")

In [ ]:
# Process multiple runs
def process_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[RunInfo]:
    runs = get_workflow_runs(repo, workflow, limit=limit)
    if not runs:
        return []
    
    print(f"\nProcessing {len(runs)} runs...\n")
    results = []
    
    for i, run in enumerate(runs, 1):
        print(f"[{i}/{len(runs)}] Run #{run['run_number']}...", end='')
        log = download_run_log(repo, run['id'], run['run_number'])
        
        if not log:
            print(" ❌ Failed")
            continue
        
        try:
            info = process_log(log, run['run_number'])
            results.append(info)
            print(f" ✓ (Q:{info.has_question} E:{info.error_flag})")
        except Exception as e:
            print(f" ⚠️  {e}")
    
    print(f"\n✅ Processed {len(results)} runs")
    return results

print("✅ process_runs()")

In [ ]:
# EXCEL OUTPUT WITH VERSIONING
def write_to_excel(run_info_list: List[RunInfo], output_file: Path) -> Path:
    """Write to VERSIONED Excel file (no overwrite!)."""
    
    # Find next version
    base = output_file.stem
    ext = output_file.suffix
    directory = output_file.parent
    
    version = 1
    while True:
        versioned = directory / f"{base}_v{version:03d}{ext}"
        if not versioned.exists():
            break
        version += 1
    
    print(f"Creating NEW file: {versioned.name}")
    
    # ALWAYS create fresh workbook
    wb = Workbook()
    ws = wb.active
    
    # Header
    headers = ['workflow_run_number', 'time_date', 'has_question', 
               'question_number', 'question_type', 'forecast_value', 'error_flag']
    ws.append(headers)
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
    
    # Data (sorted newest first)
    sorted_runs = sorted(run_info_list, key=lambda x: x.workflow_run_number, reverse=True)
    for info in sorted_runs:
        ws.append([info.workflow_run_number, info.time_date, info.has_question,
                   info.question_number, info.question_type, info.forecast_value, info.error_flag])
    
    # Column widths
    ws.column_dimensions['A'].width = 20
    ws.column_dimensions['B'].width = 20
    ws.column_dimensions['C'].width = 15
    ws.column_dimensions['D'].width = 18
    ws.column_dimensions['E'].width = 18
    ws.column_dimensions['F'].width = 50
    ws.column_dimensions['G'].width = 12
    
    directory.mkdir(parents=True, exist_ok=True)
    wb.save(versioned)
    
    print(f"✅ Saved: {versioned}")
    print(f"   Rows: {len(sorted_runs)}")
    return versioned

print("✅ write_to_excel() with VERSIONING")

In [ ]:
# RUN TEST BATCH
test_data = process_runs(REPO, WORKFLOW, limit=TEST_LIMIT)

In [ ]:
# WRITE TO EXCEL
if test_data:
    output_path = write_to_excel(test_data, OUTPUT_FILE)
    print(f"\n📊 Open file: {output_path}")
else:
    print("No data to write")

In [ ]:
# DEBUG: Inspect Excel
import glob
latest = sorted(glob.glob(str(OUTPUT_FILE.parent / "Runs and Question Numbers_v*.xlsx")))
if latest:
    wb = load_workbook(latest[-1])
    ws = wb.active
    print(f"Latest file: {Path(latest[-1]).name}")
    print("="*80)
    for row in list(ws.iter_rows(values_only=True))[:11]:  # Header + 10 rows
        print(f"{row[0]:<10} {row[1] or '':<20} {row[2]:<8} {row[3] or '':<10} {row[4] or '':<16} {str(row[5] or '')[:25]:<25} {row[6]:<8}")
else:
    print("No versioned files found")